# Utilização da extensão 'Parquet' - Transformando arquivos 'csv' em 'parquet': Criado pelo Nicholas Birochi

#### Importando bibliotecas essenciais para a Extração de Dados:

In [ ]:
# Importando bibliotecas:
from glob import glob  # Busca de arquivos por padrões...
import pandas as pd    # Análise e manipulação de dados...
import os              # Operações com arquivos e sistema...

#### Criando as funções necessárias para esse processo:

In [ ]:
# Função para a conversão de cada uma das colunas em formato "string", assim não perdendo nenhum dado:
def converter_colunas_mistas_para_str(df):
    for col in df.columns:
        if df[col].apply(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

def ler_sem_colunas(path, colunas_excluir, **kwargs):
    colunas = pd.read_csv(path, nrows=0, **kwargs).columns.tolist()

    if all(isinstance(c, int) for c in colunas_excluir):
        colunas_ler = [col for i, col in enumerate(colunas) if i not in colunas_excluir]
    else:
        colunas_ler = [col for col in colunas if col not in colunas_excluir]

    return pd.read_csv(path, usecols=colunas_ler, **kwargs)

# Função final para a transformação de cada arquivo:
def converter_csv_para_parquet(pasta_csv, tipo, colunas_excluir=None):
    # Define o caminho de saída dependendo do tipo:
    if tipo == 'E':
        pasta_parquet = r'..\..\Base de Dados Original\parquet\Executivos'
    elif tipo == 'M':
        pasta_parquet = r'..\..\Base de Dados Original\parquet\Mensalistas'
    elif tipo == 'H':
        pasta_parquet = r'..\..\Base de Dados Original\parquet\Horistas'
    else:
        raise ValueError("Tipo deve ser 'E' ou 'M' ou 'H'")
    
    # Buscando cada arquivo em formato ".csv":
    arquivos_csv = glob(os.path.join(pasta_csv, "*.csv"))
    
    # Procurando os arquivos ".csv" e usando das funções para trocar para ".parquet"
    for caminho in arquivos_csv:
        try:
            if colunas_excluir:
                df = ler_sem_colunas(caminho, colunas_excluir, encoding='latin1')
            else:
                df = pd.read_csv(caminho, encoding='latin1')
            
            # Corrigindo coluna específica:
            if 'Movimento de Colaboradores' in df.columns:
                df['Movimento de Colaboradores'] = df['Movimento de Colaboradores'].astype(str)
            
            # Convertendo todas as colunas em string:
            df = converter_colunas_mistas_para_str(df)
            
            # Pegando o nome original do arquivo sem extensão:
            nome_base = os.path.splitext(os.path.basename(caminho))[0]
            nome_arquivo = f"{nome_base}.parquet"
            caminho_salvar = os.path.join(pasta_parquet, nome_arquivo)

            # Salvando o arquivo:
            df.to_parquet(caminho_salvar, index=False, engine='pyarrow')
            print(f"✅ Arquivo salvo: {caminho_salvar}")

        except Exception as e:
            print(f"⚠️ Erro ao processar {caminho}: {e}")

#### Inserindo os Endereços e nomes para dentro das funções:

In [ ]:
# Exemplo de uso para Executivos:
converter_csv_para_parquet(
    pasta_csv=R'..\..\Base de Dados Original\csv\Executivos',
    tipo='E',
    colunas_excluir=[0, 1, 3, 8, 14, 17]
)

In [ ]:
# Exemplo de uso para Mensalistas:
converter_csv_para_parquet(
    pasta_csv=R'..\..\Base de Dados Original\csv\Mensalistas',
    tipo='M',
    colunas_excluir=[0, 1, 3, 8, 14, 17]
)

In [ ]:
# Exemplo de uso para Mensalistas:
converter_csv_para_parquet(
    pasta_csv=R'..\..\Base de Dados Original\csv\Horistas',
    tipo='H',
    colunas_excluir=[0, 1, 3, 8, 14, 17]
)